# 02 — Tiền xử lý & Feature Engineering

Mục tiêu:
- Làm sạch dữ liệu (missing, outlier, encoding)
- Tạo 7 đặc trưng mới dựa trên domain knowledge HR
- Kết nối EDA insights → feature design

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

import pandas as pd
import numpy as np

from src.data.loader import load_data, load_config
from src.data.cleaner import HRDataCleaner
from src.features.builder import FeatureBuilder

config = load_config("configs/params.yaml")

## 2.1 Làm sạch dữ liệu

In [ ]:
df = load_data(config["data"]["raw_path"])
print("Trước khi làm sạch:", df.shape)

cleaner = HRDataCleaner(target_col="Attrition")
df_clean = cleaner.clean(df)
print("Sau khi làm sạch:", df_clean.shape)

# Kiểm tra BusinessTravel đã được chuẩn hóa
print("\nBusinessTravel values:", df_clean["BusinessTravel"].unique())

## 2.2 Feature Engineering

Dựa trên EDA insights, tạo 7 features mới:

| Feature | Nguồn gốc (EDA) | Ý nghĩa |
|---|---|---|
| SatisfactionIndex | Các chỉ số hài lòng đều ảnh hưởng | Tổng hợp 4 chỉ số |
| CareerGrowthRate | Junior nghỉ nhiều | Tốc độ thăng tiến |
| PromotionStagnation | Lâu không thăng chức → nghỉ | Mức độ trì trệ |
| WorkloadScore | OT + Travel → nghỉ | Áp lực công việc |
| LoyaltyScore | Thâm niên ngắn → nghỉ | Mức trung thành |
| AvgYearsPerCompany | Nhảy việc thường xuyên | TB năm/công ty |
| IncomePerLevel | Thu nhập thấp → nghỉ | Lương so với level |

In [ ]:
fb = FeatureBuilder(df_clean)
df_featured = fb.build_all()

# Xem các features mới
new_features = ["SatisfactionIndex", "CareerGrowthRate", "PromotionStagnation",
                "WorkloadScore", "LoyaltyScore", "AvgYearsPerCompany", "IncomePerLevel"]
df_featured[new_features].describe().round(2)

## 2.3 Mã hóa cho mô hình ML

In [ ]:
df_encoded = cleaner.encode(df_featured)
print(f'Shape sau mã hóa: {df_encoded.shape}')
df_encoded.head()

## 2.4 Rời rạc hóa cho Association Rules

In [ ]:
df_disc = cleaner.discretize_for_mining(df_clean)
df_disc.head()

## 2.5 Lưu dữ liệu đã xử lý

In [ ]:
cleaner.save_processed(df_clean, config["data"]["processed_path"])
print("Done!")